In [1]:
#pip install nimare

In [1]:
import nilearn as nl
import numpy as np
import matplotlib
import pandas as pd
import os, requests
import nibabel as nib
import matplotlib.pyplot as plt
import openpyxl as xl
import os
import subprocess

alias_dir = "/Users/silvycollin/Documents/research/narratives/fmriprep/"

from nilearn import datasets, image, masking
import numpy as np

import nimare

try:
    # Newer NiMARE versions
    from nimare.datasets import fetch_neurosynth
except ImportError:
    # Older NiMARE versions
    from nimare.extract import fetch_neurosynth



In [2]:

# ----------------------------
# 2. Load Harvard-Oxford cortical atlas
# ----------------------------
atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')
atlas_img = image.load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
labels = atlas.labels

# Select medial and temporal regions
target_regions = [
    "Middle Temporal Gyrus, anterior division",
    "Superior Temporal Gyrus, anterior division",
]

# Identify indices for selected regions
indices = [i for i, name in enumerate(labels) if any(r in name for r in target_regions)]
print("Selected HO indices:", indices)
print("Selected labels:", [labels[i] for i in indices])

# Create anatomical mask
anat_mask_data = np.isin(atlas_data, indices).astype(int)
anat_mask_img = image.new_img_like(atlas_img, anat_mask_data)
anat_mask_img.to_filename("antMTG_STG_HO_mask.nii.gz")
print("Saved anatomical medial+temporal mask as Medial_Temporal_HO_mask.nii.gz")

# ----------------------------
# 4. Split into left / right hemispheres
# ----------------------------
affine = anat_mask_img.affine
coords = np.indices(anat_mask_data.shape)
# Compute MNI x-coordinates
x_coords = affine[0, 0] * coords[0] + affine[0, 3]

left_mask = (anat_mask_data > 0) & (x_coords < 0)
right_mask = (anat_mask_data > 0) & (x_coords > 0)

left_img = image.new_img_like(anat_mask_img, left_mask.astype(int))
right_img = image.new_img_like(anat_mask_img, right_mask.astype(int))

left_img.to_filename("antSTG_MTG_HOonly_L.nii.gz")
right_img.to_filename("antSTG_MTG_HOonly_R.nii.gz")

print("Left voxels:", np.sum(left_mask), "Right voxels:", np.sum(right_mask))


[get_dataset_dir] Dataset found in /Users/silvycollin/nilearn_data/fsl

Selected HO indices: [9, 11]
Selected labels: ['Superior Temporal Gyrus, anterior division', 'Middle Temporal Gyrus, anterior division']
Saved anatomical medial+temporal mask as Medial_Temporal_HO_mask.nii.gz
Left voxels: 699 Right voxels: 670


/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_1934/4101434703.py:22: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  anat_mask_img = image.new_img_like(atlas_img, anat_mask_data)
/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_1934/4101434703.py:37: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  left_img = image.new_img_like(anat_mask_img, left_mask.astype(int))
/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_1934/4101434703.py:38: UserWarning: Data array used to create a n